In [1]:
!pip install -q kagglehub xgboost networkx plotly

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

!pip install -q torch-geometric

PyTorch version: 2.10.0+cpu
CUDA available: False
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import plotly.graph_objects as go
import os, glob, warnings, time
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_recall_curve,
    average_precision_score, roc_auc_score, f1_score, precision_score,
    recall_score, roc_curve, auc
)

import torch
import torch.nn.functional as F

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'All imports loaded! Device: {DEVICE}')

All imports loaded! Device: cpu


In [ ]:
import kagglehub

print("Downloading IBM AML dataset...")
ibm_path = kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml")
print(f"  Path: {ibm_path}")
for f in sorted(os.listdir(ibm_path)):
    sz = os.path.getsize(os.path.join(ibm_path, f)) / (1024*1024)
    print(f"    {f} ({sz:.1f} MB)")

print("\nDownloading Czech Financial dataset...")
czech_path = kagglehub.dataset_download("siavashraz/1999-czech-financial-dataset")
print(f"  Path: {czech_path}")
for root, dirs, files in os.walk(czech_path):
    for f in files:
        fp = os.path.join(root, f)
        sz = os.path.getsize(fp) / (1024*1024)
        print(f"    {os.path.relpath(fp, czech_path)} ({sz:.1f} MB)")

In [ ]:
ibm_raw = pd.read_csv(os.path.join(ibm_path, 'HI-Small_Trans.csv'))

print("="*65)
print("IBM AML DATASET (HI-Small_Trans.csv)")
print("="*65)
print(f"Shape: {ibm_raw.shape[0]:,} rows × {ibm_raw.shape[1]} columns")
print(f"Memory: {ibm_raw.memory_usage(deep=True).sum()/1e6:.1f} MB")
print(f"\nColumns & types:")
for col in ibm_raw.columns:
    print(f"  {col:<25} {str(ibm_raw[col].dtype):<10} nunique={ibm_raw[col].nunique():,}")

vc = ibm_raw['Is Laundering'].value_counts()
print(f"\n🎯 Target — Is Laundering:")
print(f"   Clean (0):      {vc[0]:>10,} ({vc[0]/len(ibm_raw)*100:.2f}%)")
print(f"   Laundering (1): {vc[1]:>10,} ({vc[1]/len(ibm_raw)*100:.2f}%)")
print(f"   Imbalance:      1:{vc[0]//vc[1]}")
print(f"\nMissing values: {ibm_raw.isnull().sum().sum()}")
print(f"Duplicate rows:  {ibm_raw.duplicated().sum()}")
ibm_raw.head(3)

In [ ]:
def find_csv(base, name):
    results = glob.glob(os.path.join(base, '**', name), recursive=True)
    return results[0] if results else None

czech_tables = {}
for fname in ['trans.csv','account.csv','client.csv','disp.csv',
              'district.csv','loan.csv','order.csv','card.csv']:
    fpath = find_csv(czech_path, fname)
    if fpath:
        tname = fname.split('.')[0]
        czech_tables[tname] = pd.read_csv(fpath, sep=';', low_memory=False)
        print(f"{fname:<15} → {czech_tables[tname].shape[0]:>8,} rows × {czech_tables[tname].shape[1]} cols")

cz_trans_raw = czech_tables['trans']
cz_loan = czech_tables['loan']
cz_account = czech_tables['account']

print(f"\n" + "="*65)
print("CZECH TRANSACTIONS (trans.csv)")
print("="*65)
print(f"Shape: {cz_trans_raw.shape[0]:,} rows × {cz_trans_raw.shape[1]} columns")
for col in cz_trans_raw.columns:
    print(f"  {col:<15} {str(cz_trans_raw[col].dtype):<10} nunique={cz_trans_raw[col].nunique():,}")

print(f"\nCzech Loan Status (risk proxy):")
for status, count in cz_loan['status'].value_counts().items():
    desc = {'A':'finished OK','B':'finished NOT paid','C':'running OK','D':'running IN DEBT'}[status]
    print(f"   {status}: {count:>4} — {desc}")

cz_trans_raw.head(3)

In [ ]:
print("IBM — Numerical Summary:")
display(ibm_raw.describe())
print("\nCzech — Numerical Summary:")
display(cz_trans_raw.describe())

In [ ]:
ibm = ibm_raw.copy()

# Clean amount columns
for col in ['Amount Received', 'Amount Paid']:
    if ibm[col].dtype == 'object':
        ibm[col] = ibm[col].astype(str).str.replace('[^0-9.]', '', regex=True)
    ibm[col] = pd.to_numeric(ibm[col], errors='coerce').fillna(0)

# Parse timestamps
ibm['Timestamp'] = pd.to_datetime(ibm['Timestamp'], errors='coerce')

# Create unique entity IDs
ibm['sender_id'] = ibm['From Bank'].astype(str) + '_' + ibm['Account'].astype(str)
ibm['receiver_id'] = ibm['To Bank'].astype(str) + '_' + ibm['Account.1'].astype(str)

# Remove duplicates
n0 = len(ibm)
ibm = ibm.drop_duplicates()
print(f"IBM cleaned: {n0:,} → {len(ibm):,} (removed {n0-len(ibm)} duplicates)")
print(f"  Amount range: {ibm['Amount Paid'].min():.2f} — {ibm['Amount Paid'].max():.2f}")
print(f"  Date range:   {ibm['Timestamp'].min()} → {ibm['Timestamp'].max()}")
print(f"  Unique senders:   {ibm['sender_id'].nunique():,}")
print(f"  Unique receivers: {ibm['receiver_id'].nunique():,}")

In [ ]:
cz = cz_trans_raw.copy()

# Parse date
cz['date'] = pd.to_datetime(cz['date'], format='%Y%m%d', errors='coerce')
if cz['date'].isnull().sum() > len(cz) * 0.5:
    cz['date'] = pd.to_datetime(cz_trans_raw['date'].astype(str), errors='coerce')

# Clean amounts
cz['amount'] = pd.to_numeric(cz['amount'], errors='coerce').fillna(0)
cz['balance'] = pd.to_numeric(cz['balance'], errors='coerce').fillna(0)

# Fill missing categoricals
for col in ['type', 'operation', 'k_symbol']:
    if col in cz.columns:
        cz[col] = cz[col].fillna('unknown')

# Attach loan risk (EVALUATION ONLY)
cz_risk = cz_loan[['account_id', 'status']].copy()
cz_risk['loan_risky'] = cz_risk['status'].isin(['B', 'D']).astype(int)
cz = cz.merge(cz_risk[['account_id', 'loan_risky']], on='account_id', how='left')
cz['loan_risky'] = cz['loan_risky'].fillna(-1).astype(int)

n0 = len(cz)
cz = cz.drop_duplicates()
print(f"Czech cleaned: {n0:,} → {len(cz):,} (removed {n0-len(cz)} duplicates)")
print(f"  Amount range: {cz['amount'].min():.2f} — {cz['amount'].max():.2f}")
print(f"  Date range:   {cz['date'].min()} → {cz['date'].max()}")
print(f"  With loan label: {(cz['loan_risky']>=0).sum():,} | Without: {(cz['loan_risky']==-1).sum():,}")

In [ ]:
# ── Harmonize IBM ──
ibm_h = pd.DataFrame({
    'amount': ibm['Amount Paid'],
    'amount_received': ibm['Amount Received'],
    'timestamp': ibm['Timestamp'],
    'sender_id': 'IBM_' + ibm['sender_id'],
    'receiver_id': 'IBM_' + ibm['receiver_id'],
    'payment_type': ibm['Payment Format'].astype(str),
    'currency_send': ibm['Payment Currency'].astype(str),
    'currency_recv': ibm['Receiving Currency'].astype(str),
    'source': 'ibm',
    '_label': ibm['Is Laundering'],
})

# ── Harmonize Czech ──
cz_recv = 'CZ_bank_' + cz['bank'].astype(str) if 'bank' in cz.columns else pd.Series('CZ_unknown', index=cz.index)

cz_h = pd.DataFrame({
    'amount': cz['amount'].abs(),
    'amount_received': cz['balance'],
    'timestamp': cz['date'],
    'sender_id': 'CZ_' + cz['account_id'].astype(str),
    'receiver_id': cz_recv if isinstance(cz_recv, pd.Series) else pd.Series(cz_recv, index=cz.index),
    'payment_type': cz['type'].astype(str) if 'type' in cz.columns else 'unknown',
    'currency_send': 'CZK',
    'currency_recv': 'CZK',
    'source': 'czech',
    '_label': cz['loan_risky'],
})

# ── Concatenate ──
merged = pd.concat([ibm_h, cz_h], ignore_index=True)

print("="*65)
print("MERGED DATASET")
print("="*65)
print(f"Total:             {len(merged):,} transactions")
print(f"  IBM:             {(merged['source']=='ibm').sum():,}")
print(f"  Czech:           {(merged['source']=='czech').sum():,}")
print(f"Unique senders:    {merged['sender_id'].nunique():,}")
print(f"Unique receivers:  {merged['receiver_id'].nunique():,}")
print(f"Unique edges:      {merged.groupby(['sender_id','receiver_id']).ngroup().nunique():,}")

In [ ]:
ibm_d = merged[merged['source']=='ibm']
cz_d = merged[merged['source']=='czech']

# ══════════════════════════════════════════════════════════════
# VIZ 1: Dataset Composition & Class Balance
# ══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Source split
ax = axes[0]
src = merged['source'].value_counts()
ax.pie(src.values, labels=[f"{k.upper()}\n({v:,})" for k,v in src.items()],
       colors=['#3498db','#e67e22'], autopct='%1.1f%%', startangle=90, textprops={'fontsize':12})
ax.set_title('Dataset Composition', fontsize=14, fontweight='bold')

# IBM target
ax = axes[1]
lbl = ibm_d['_label'].value_counts().sort_index()
bars = ax.bar(['Clean (0)','Laundering (1)'], lbl.values,
              color=['#2ecc71','#e74c3c'], edgecolor='black', linewidth=0.5)
for b,v in zip(bars, lbl.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()*1.01, f'{v:,}', ha='center', fontsize=11)
ax.set_title('IBM — Labels (eval only)', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')

# Czech risk
ax = axes[2]
cl = cz_d['_label'].value_counts().sort_index()
nm = {-1:'No Loan Info', 0:'Loan OK', 1:'Loan Default'}
clr = {-1:'#95a5a6', 0:'#2ecc71', 1:'#e74c3c'}
bars = ax.bar([nm[k] for k in cl.index], cl.values, color=[clr[k] for k in cl.index],
              edgecolor='black', linewidth=0.5)
for b,v in zip(bars, cl.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()*1.01, f'{v:,}', ha='center', fontsize=11)
ax.set_title('Czech — Loan Risk (eval proxy)', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.savefig('viz1_composition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
# VIZ 2: Amount Distributions
# ══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# IBM clean vs laundering
ax = axes[0,0]
ax.hist(np.log1p(ibm_d[ibm_d['_label']==0]['amount']), bins=60, alpha=0.6, color='#2ecc71', label='Clean', density=True)
ax.hist(np.log1p(ibm_d[ibm_d['_label']==1]['amount']), bins=60, alpha=0.6, color='#e74c3c', label='Laundering', density=True)
ax.set_title('IBM — Amount by Label (log scale)', fontsize=13, fontweight='bold')
ax.set_xlabel('log(1+Amount)'); ax.legend()

# Czech by risk
ax = axes[0,1]
czl = cz_d[cz_d['_label']>=0]
if len(czl) > 0:
    ax.hist(np.log1p(czl[czl['_label']==0]['amount']), bins=60, alpha=0.6, color='#2ecc71', label='Loan OK', density=True)
    ax.hist(np.log1p(czl[czl['_label']==1]['amount']), bins=60, alpha=0.6, color='#e74c3c', label='Loan Default', density=True)
ax.set_title('Czech — Amount by Risk (log scale)', fontsize=13, fontweight='bold')
ax.set_xlabel('log(1+Amount)'); ax.legend()

# Distribution shift
ax = axes[1,0]
ax.hist(np.log1p(ibm_d['amount'].clip(lower=0)), bins=60, alpha=0.5, color='#3498db', label='IBM', density=True)
ax.hist(np.log1p(cz_d['amount'].clip(lower=0)), bins=60, alpha=0.5, color='#e67e22', label='Czech', density=True)
ax.set_title('Distribution Shift — IBM vs Czech', fontsize=13, fontweight='bold')
ax.set_xlabel('log(1+Amount)'); ax.legend()

# Box plot comparison
ax = axes[1,1]
data_bp = [
    ibm_d[ibm_d['_label']==0]['amount'].clip(upper=ibm_d['amount'].quantile(0.99)),
    ibm_d[ibm_d['_label']==1]['amount'].clip(upper=ibm_d['amount'].quantile(0.99)),
    cz_d['amount'].clip(upper=cz_d['amount'].quantile(0.99)),
]
bp = ax.boxplot(data_bp, labels=['IBM Clean','IBM Laundering','Czech'], patch_artist=True)
for p,c in zip(bp['boxes'], ['#2ecc71','#e74c3c','#e67e22']):
    p.set_facecolor(c); p.set_alpha(0.7)
ax.set_title('Amount Boxplot Comparison (99th pctl)', fontsize=13, fontweight='bold')
ax.set_ylabel('Amount')

plt.tight_layout()
plt.savefig('viz2_amounts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
# VIZ 3: Time Patterns
# ══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# IBM daily
ax = axes[0,0]
ibm_daily = ibm_d.set_index('timestamp').resample('D').agg(total=('amount','count'), launder=('_label','sum')).dropna()
ax.plot(ibm_daily.index, ibm_daily['total'], color='#3498db', alpha=0.8)
ax2 = ax.twinx()
ax2.fill_between(ibm_daily.index, ibm_daily['launder'], color='#e74c3c', alpha=0.3)
ax.set_title('IBM — Daily Volume', fontsize=13, fontweight='bold')
ax.set_ylabel('Total', color='#3498db'); ax2.set_ylabel('Laundering', color='#e74c3c')
ax.tick_params(axis='x', rotation=30)

# Czech monthly
ax = axes[0,1]
cz_monthly = cz_d.dropna(subset=['timestamp']).set_index('timestamp').resample('M').size()
ax.plot(cz_monthly.index, cz_monthly.values, color='#e67e22', linewidth=2)
ax.set_title('Czech — Monthly Volume', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=30)

# IBM hourly
ax = axes[1,0]
ibm_hourly = ibm_d.groupby([ibm_d['timestamp'].dt.hour,'_label']).size().unstack(fill_value=0)
ibm_hourly.plot(ax=ax, color=['#2ecc71','#e74c3c'], linewidth=2)
ax.set_title('IBM — Hourly Pattern', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour'); ax.legend(['Clean','Laundering'])

# Payment types
ax = axes[1,1]
pt = ibm_d.groupby(['payment_type','_label']).size().unstack(fill_value=0)
pt.plot(kind='bar', ax=ax, color=['#2ecc71','#e74c3c'], edgecolor='black', linewidth=0.3)
ax.set_title('IBM — Payment Type by Label', fontsize=13, fontweight='bold')
ax.legend(['Clean','Laundering']); ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('viz3_time.png', dpi=150, bbox_inches='tight')
plt.show()